In [1]:
!pip install -q "huggingface_hub>=0.26.2" polars albumentations pyarrow

In [2]:

from pathlib import Path
import polars as pl
from collections import Counter
from tqdm import tqdm
import shutil
import math

# Dùng trực tiếp path Kaggle
BASE_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset")
TRAIN_DIR = BASE_DIR / "train"
SILVER_DIR = BASE_DIR / "silver"   # dùng sau nếu cần
TEST_DIR = BASE_DIR / "test"

print("Train dir:", TRAIN_DIR)
print("Silver dir:", SILVER_DIR)
print("Test dir:", TEST_DIR)

train_meta_path = TRAIN_DIR / "metadata.jsonl"
df = pl.read_ndjson(train_meta_path)
print(df.head())
print("Num train images:", df.height)

OUT_ROOT = Path("/kaggle/working/layout_data/rukopys")
(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

TYPE2ID = {
    "handwritten": 0,
    "printed": 1,
    "formula": 2,
    "table": 3,
    "annotation": 4,
    "image": 5,
    "graph": 6,
}

def convert_bbox_xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return cx, cy, w, h

class_counts = Counter()
source_counts = Counter()
image_entries = []

RARE_CLASSES = {"table", "graph", "annotation", "formula"}

for row in tqdm(df.iter_rows(named=True)):
    # tên cột đúng trong metadata: file_name, image_width, image_height
    fname_rel = row["file_name"]      # ví dụ: "images/09ba34d....jpg"
    src       = row["source"]         # dictation / archive / university / school
    img_w     = row["image_width"]
    img_h     = row["image_height"]
    regions   = row["regions"] or []

    source_counts[src] += 1

    # đường dẫn thật tới file ảnh trong Kaggle dataset
    in_path = TRAIN_DIR / fname_rel   # train/images/xxxx.jpg
    if not in_path.exists():
        continue

    # chỉ copy basename sang OUT_ROOT/images
    fname = Path(fname_rel).name
    out_img = OUT_ROOT / "images" / fname
    if not out_img.exists():
        shutil.copy(in_path, out_img)

    label_lines = []
    rare_score = 0
    has_rare = False

    for r in regions:
        t = r["type"]
        if t not in TYPE2ID:
            continue
        x1, y1, x2, y2 = r["bbox"]

        x1 = max(0, min(x1, img_w - 1))
        x2 = max(0, min(x2, img_w - 1))
        y1 = max(0, min(y1, img_h - 1))
        y2 = max(0, min(y2, img_h - 1))
        if x2 <= x1 or y2 <= y1:
            continue

        cx, cy, w, h = convert_bbox_xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h)
        if w <= 0 or h <= 0:
            continue

        cid = TYPE2ID[t]
        label_lines.append(f"{cid} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        class_counts[t] += 1

        if t in RARE_CLASSES:
            rare_score += 1
            has_rare = True

    label_path = OUT_ROOT / "labels" / (Path(fname).stem + ".txt")
    with open(label_path, "w", encoding="utf-8") as f:
        f.write("\n".join(label_lines))

    image_entries.append({
        "filename": fname,   # chỉ còn basename, dùng về sau
        "source": src,
        "rare_score": rare_score,
        "has_rare": has_rare,
    })

print("Class counts:", class_counts)
print("Source counts:", source_counts)

Train dir: /kaggle/input/datasets/quii29/rukopys-dataset/train
Silver dir: /kaggle/input/datasets/quii29/rukopys-dataset/silver
Test dir: /kaggle/input/datasets/quii29/rukopys-dataset/test
shape: (5, 7)
┌────────────────┬─────────────┬──────────────┬───────────┬────────────────┬──────┬────────────────┐
│ file_name      ┆ image_width ┆ image_height ┆ source    ┆ annotation_sou ┆ year ┆ regions        │
│ ---            ┆ ---         ┆ ---          ┆ ---       ┆ rce            ┆ ---  ┆ ---            │
│ str            ┆ i64         ┆ i64          ┆ str       ┆ ---            ┆ i64  ┆ list[struct[5] │
│                ┆             ┆              ┆           ┆ str            ┆      ┆ ]              │
╞════════════════╪═════════════╪══════════════╪═══════════╪════════════════╪══════╪════════════════╡
│ images/09ba34d ┆ 3468        ┆ 4624         ┆ dictation ┆ annotator      ┆ 2024 ┆ [{[1213, 350,  │
│ f-2665-452c-9e ┆             ┆              ┆           ┆                ┆      ┆ … 530]

1330it [00:46, 28.90it/s]

Class counts: Counter({'handwritten': 21523, 'formula': 2950, 'annotation': 484, 'printed': 295, 'image': 117, 'table': 111, 'graph': 24})
Source counts: Counter({'school': 682, 'dictation': 359, 'university': 162, 'archive': 127})


In [3]:
from sklearn.model_selection import train_test_split

filenames = [e["filename"] for e in image_entries]
sources   = [e["source"]   for e in image_entries]

train_files, val_files = train_test_split(
    filenames,
    test_size=0.15,
    random_state=42,
    stratify=sources,
)

train_set = set(train_files)
val_set   = set(val_files)

# Đếm source trong train
source_train_counts = Counter(e["source"] for e in image_entries if e["filename"] in train_set)
print("Train source counts:", source_train_counts)

total_train = sum(source_train_counts.values())
src_weights = {}
for s, c in source_train_counts.items():
    p = c / total_train
    target_p = 1.0 / 4.0
    src_weights[s] = target_p / p

print("Source weights:", src_weights)

oversampled_train = []
for e in image_entries:
    fname = e["filename"]
    if fname not in train_set:
        continue
    src = e["source"]
    base_w = src_weights.get(src, 1.0)

    if e["has_rare"]:
        rare_factor = 1.0 + 0.5 * e["rare_score"]
    else:
        rare_factor = 1.0

    w = base_w * rare_factor
    repeats = min(int(math.ceil(w)), 5)
    oversampled_train.extend([fname] * repeats)

print("Num oversampled train entries:", len(oversampled_train))

Train source counts: Counter({'school': 579, 'dictation': 305, 'university': 138, 'archive': 108})
Source weights: {'dictation': 0.9262295081967213, 'archive': 2.615740740740741, 'university': 2.0471014492753623, 'school': 0.4879101899827288}
Num oversampled train entries: 2346


In [4]:
train_txt = OUT_ROOT / "train.txt"
val_txt   = OUT_ROOT / "val.txt"

with open(train_txt, "w", encoding="utf-8") as f:
    for fname in oversampled_train:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

with open(val_txt, "w", encoding="utf-8") as f:
    for fname in val_set:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

print("Wrote", train_txt, "and", val_txt)

Wrote /kaggle/working/layout_data/rukopys/train.txt and /kaggle/working/layout_data/rukopys/val.txt


In [5]:
import os
import yaml
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

OUT_ROOT = Path("/kaggle/working/layout_data/rukopys")

# 1) Tải checkpoint pretrained
CKPT_PATH = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt",
)
print("Checkpoint:", CKPT_PATH)

# 2) Ghi file YAML cấu hình ra /kaggle/working/
yaml_path = "/kaggle/working/rukopys_dataset.yaml"
data_config = {
    "path": str(OUT_ROOT),
    "train": "train.txt",   
    "val": "val.txt",       
    "nc": 7,
    "names": ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
}

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_config, f, sort_keys=False)

shutil.copy(yaml_path, yaml_path + ".yaml")

# 3) Clone repo 
%cd /kaggle/working
if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git
%cd DocLayout-YOLO
!pip install -q -e .

# --- BƯỚC MỚI: VÁ LỖI PYTORCH 2.6 TRỰC TIẾP VÀO MÃ NGUỒN ---
checks_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/checks.py"
if os.path.exists(checks_file):
    with open(checks_file, "r", encoding="utf-8") as f:
        code = f.read()
    
    # Ép hàm check_amp luôn trả về True, bỏ qua bài test tải file
    if "def check_amp(model):" in code and "return True" not in code.split("def check_amp(model):")[1][:20]:
        code = code.replace("def check_amp(model):", "def check_amp(model):\n    return True\n")
        with open(checks_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("Đã vá (patch) thành công hàm check_amp!")

# Gỡ bỏ ray để chặn lỗi "is_session_enabled" ở cuối epoch
!pip uninstall ray -y 

# 4) Chạy train
EPOCHS = 80
BATCH_SIZE = 8  
IMGSZ = 1280
LR0 = 0.002

# Đã xóa --exist-ok ở cuối
!WANDB_MODE=disabled RAY_DISABLE_TUNE=1 python train.py \
  --data /kaggle/working/rukopys_dataset.yaml \
  --model doclayout_yolo_small \
  --epoch {EPOCHS} \
  --image-size {IMGSZ} \
  --batch-size {BATCH_SIZE} \
  --project rukopys_ft_yolo_small \
  --optimizer Adam \
  --lr0 {LR0} \
  --warmup-epochs 1.0 \
  --patience 8 \
  --pretrain {CKPT_PATH} \
  --device 0,1 \
  --workers 8

doclayout_yolo_docstructbench_imgsz1024.(…):   0%|          | 0.00/40.7M [00:00<?, ?B/s]

Checkpoint: /root/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt
/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 31.11 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for doclayout_yolo (pyproject.toml) ... done
Đã vá (patch) thành công hàm check_amp!
Found existing installation: ray 2.54.0
Uninstalling ray-2.54.0:
  Successfully uninstalled ray-2.54.0
N